# 01 — Train an Iris Model and Create Deployment Artifacts

## Goal

This notebook is the **offline build stage** of an ML deployment workflow. It trains and validates an Iris classifier, then exports the exact files that an online FastAPI service loads.

```text
TRAINING / BUILD TIME                      SERVING / RUN TIME
─────────────────────                      ──────────────────
Iris dataset                               HTTP request
    ↓                                          ↓
feature contract                           Pydantic validation
    ↓                                          ↓
train/test split                           serialized Pipeline
    ↓                                          ↓
preprocessing + model                      prediction
    ↓                                          ↓
evaluation                                 JSON response
    ↓
serialized artifacts
```

**Critical rule:** the API must not retrain the model. Training produces deployment artifacts; serving consumes them.

## 1. Environment and reproducibility

The project uses **Conda**. The repository-level `environment.yml` is the environment contract. The output below records the validation runtime used to execute this committed notebook.

In [1]:
from pathlib import Path
import json
import platform
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print(f"Python:       {platform.python_version()}")
print(f"NumPy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"joblib:       {joblib.__version__}")

Python:       3.13.5
NumPy:        2.3.5
pandas:       2.2.3
scikit-learn: 1.8.0
joblib:       1.5.3


## 2. Project paths and artifacts

The notebook writes generated files to `artifacts/`. The FastAPI service reads only these files.

```text
notebook ──builds──► iris_model.joblib
                 ├─► model_metadata.json
                 └─► metrics.json
```

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "iris_model.joblib"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"
METRICS_PATH = ARTIFACT_DIR / "metrics.json"

print("Artifact directory:", ARTIFACT_DIR)

Artifact directory: /mnt/data/awesome-api-resources/examples/01-iris-fastapi-local-serving/artifacts


## 3. Load Iris and define the API feature contract

We rename scikit-learn's labels to stable JSON-friendly names. **The API request schema and model DataFrame must use the same names and order.**

In [3]:
iris = load_iris(as_frame=True)
raw_df = iris.frame.copy()

FEATURE_NAMES = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
CLASS_NAMES = [str(name) for name in iris.target_names]

X = raw_df.drop(columns=["target"]).copy()
X.columns = FEATURE_NAMES
y = raw_df["target"].copy()

print("Rows:", len(raw_df))
print("Features:", FEATURE_NAMES)
print("Classes:", CLASS_NAMES)
X.head()

Rows: 150
Features: ['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']
Classes: ['setosa', 'versicolor', 'virginica']


   sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm
0              5.1             3.5              1.4             0.2
1              4.9             3.0              1.4             0.2
2              4.7             3.2              1.3             0.2
3              4.6             3.1              1.5             0.2
4              5.0             3.6              1.4             0.2

## 4. Split data reproducibly

`stratify=y` preserves class balance and `random_state=42` makes the split deterministic.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 120
Test rows: 30


## 5. Build one deployable Pipeline

The serialized artifact includes preprocessing **and** the estimator:

```text
raw features → StandardScaler → LogisticRegression → prediction
```

This is how we prevent training-serving preprocessing drift.

In [5]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 6. Evaluate before packaging

A model should not become a deployment artifact without an explicit quality check.

In [6]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

Test accuracy: 0.9333

              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        10
  versicolor     0.9000    0.9000    0.9000        10
   virginica     0.9000    0.9000    0.9000        10

    accuracy                         0.9333        30
   macro avg     0.9333    0.9333    0.9333        30
weighted avg     0.9333    0.9333    0.9333        30


In [7]:
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f"actual_{name}" for name in CLASS_NAMES],
    columns=[f"pred_{name}" for name in CLASS_NAMES],
)
confusion

                   pred_setosa  pred_versicolor  pred_virginica
actual_setosa               10                0               0
actual_versicolor            0                9               1
actual_virginica             0                1               9

## 7. Create metadata and metrics

A binary model without operational context is incomplete. We package the model identity, version, ordered feature schema, class names, and evaluation results beside the binary artifact.

In [8]:
MODEL_VERSION = "1.0.0"

metadata = {
    "model_name": "iris-logistic-regression",
    "model_version": MODEL_VERSION,
    "framework": "scikit-learn",
    "scikit_learn_version": sklearn.__version__,
    "feature_names": FEATURE_NAMES,
    "class_names": CLASS_NAMES,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

report = classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, output_dict=True
)
metrics = {
    "accuracy": float(accuracy),
    "classification_report": report,
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}

print(json.dumps({k: v for k, v in metadata.items() if k != "created_at_utc"}, indent=2))

{
  "model_name": "iris-logistic-regression",
  "model_version": "1.0.0",
  "framework": "scikit-learn",
  "scikit_learn_version": "1.8.0",
  "feature_names": [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
  ],
  "class_names": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "training_rows": 120,
  "test_rows": 30
}


## 8. Serialize the deployment artifacts

`joblib.dump()` writes the fitted Pipeline. JSON keeps metadata and metrics inspectable by humans and deployment tooling.

In [9]:
joblib.dump(model, MODEL_PATH)
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

for path in (MODEL_PATH, METADATA_PATH, METRICS_PATH):
    print(f"{path.name:<24} {path.stat().st_size:>8} bytes")

iris_model.joblib            1889 bytes
model_metadata.json          426 bytes
metrics.json                 922 bytes


## 9. Deployment check — reload from disk

This is the important boundary test. We stop trusting the in-memory training object and load the serialized artifact exactly as FastAPI will.

In [10]:
reloaded_model = joblib.load(MODEL_PATH)

api_like_sample = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=FEATURE_NAMES,
)

class_id = int(reloaded_model.predict(api_like_sample)[0])
probability_values = reloaded_model.predict_proba(api_like_sample)[0]

result = {
    "prediction": CLASS_NAMES[class_id],
    "class_id": class_id,
    "probabilities": {
        class_name: round(float(probability), 6)
        for class_name, probability in zip(CLASS_NAMES, probability_values)
    },
    "model_version": MODEL_VERSION,
}

assert result["prediction"] == "setosa"
assert abs(sum(result["probabilities"].values()) - 1.0) < 1e-5
result

{'prediction': 'setosa',
 'class_id': 0,
 'probabilities': {'setosa': 0.980813,
  'versicolor': 0.019187,
  'virginica': 0.0},
 'model_version': '1.0.0'}

## 10. Handoff to FastAPI

The notebook's responsibility ends here.

```text
POST /predict
      ↓
Pydantic validates JSON
      ↓
app.py creates the ordered DataFrame
      ↓
iris_model.joblib
      ↓
predict() + predict_proba()
      ↓
JSON response
```

From the example directory:

```bash
python test_api.py
uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```

Open `http://127.0.0.1:8000/docs`.

### Cloud/deployment mapping

```text
trained artifact → FastAPI → container → registry → cloud runtime → monitoring/autoscaling
```

The model-serving contract stays the same as infrastructure becomes more sophisticated.